# Snapshot eval statistics tables — CSV + LaTeX export across eval settings

Companion notebook to `snapshot_eval_catplot_algo_compare.ipynb`: it picks up the
**same sources** (keep the `SOURCES` block in sync) and exports **one statistics
table per metric**, with one row per (algorithm, variant) and **all eval settings
side by side as columns** — where the catplot notebook draws a single setting
(`EVAL_CFG`), this notebook tables every setting kept by `EVAL_CFGS` (default: all
discovered).

Running the export cell writes per metric into **every** source snapshot's `stat/`
directory as `strip_stats_<metric>_evalcfg_compare_<algos>.{csv,tex}`:

- a **CSV** with a flat `<setting>_<stat>` header (full precision: count, mean, std,
  median, min, max per setting), and
- a **LaTeX table body** (`.tex`) — one `mean ± std` and one `median` column per eval
  setting, rounded to 2 decimals, `\midrule` between the algorithm groups — meant to
  be `\input{}` into a hand-written booktabs `tabular` skeleton (a commented example
  sits at the top of the file), so rounding and ± assembly live in code while the
  LaTeX wrapper stays a thin, readable shell.

The body **ends with `\bottomrule`**: do NOT add another one after the `\input{}` —
LaTeX inserts file-hook tokens at the end of an `\input{}`ed file which start a new
table cell, so any `\midrule`/`\bottomrule` following `\input{}` fails with
*Misplaced `\noalign`*. Close the skeleton with `\end{tabular}` directly.

**Main input:** set `SOURCES`, `EVAL_CFGS` and `METRICS` in the next cell.


In [1]:
# --- Main input ---------------------------------------------------------------------
# One source per (algorithm, training variant) — keep SOURCES in sync with
# snapshot_eval_catplot_algo_compare.ipynb (the catplot notebook draws ONE eval
# setting; this notebook tables the eval settings side by side).
# Keys per entry:
#   path          snapshot dir (absolute, or relative to the repo root)
#   eval-kind     "eval-ray" | "eval-sb" | "eval-rbc" — which runs/eval_* dirs to use
#   algo          first row key (also used in the export filenames)
#   variant       training-variant label, the second row key
#   runs          (optional) restrict to these runs/<name> directories; default: all
#   rbc-strategy  (eval-rbc only) which rule-based strategy to use

SOURCES = [
    # --- PPO -------------------------------------------------------------------
    {
        "path": "snapshots/20260714_235444_gen_battery_lin_so_only_17kW_ppo",
        "eval-kind": "eval-ray",
        "algo": "PPO",
        "variant": "trained on 17 kWh",
    },
    {
        "path": "snapshots/20260714_235342_gen_battery_lin_so_only_15kW_ppo",
        "eval-kind": "eval-ray",
        "algo": "PPO",
        "variant": "trained on 15 kWh",
    },
    {
        "path": "snapshots/20260714_235542_gen_battery_lin_so_all_ppo",
        "eval-kind": "eval-ray",
        "algo": "PPO",
        "variant": "5-way gen.",
    },
    {
        "path": "snapshots/20260714_235647_gen_battery_lin_capacity_sampled_ppo",
        "eval-kind": "eval-ray",
        "algo": "PPO",
        "variant": "gen. distribution",
    },
    # --- SAC -------------------------------------------------------------------
    {
        "path": "snapshots/20260714_161830_gen_battery_lin_so_only_17kW",
        "eval-kind": "eval-ray",
        "algo": "SAC",
        "variant": "trained on 17 kWh",
    },
    {
        "path": "snapshots/20260714_235241_gen_battery_lin_so_only_15kW",
        "eval-kind": "eval-ray",
        "algo": "SAC",
        "variant": "trained on 15 kWh",
    },
    {
        "path": "snapshots/20260714_162614_gen_battery_lin_so_all",
        "eval-kind": "eval-ray",
        "algo": "SAC",
        "variant": "5-way gen.",
    },
    {
        "path": "snapshots/20260714_165943_gen_battery_lin_capacity_sampled",
        "eval-kind": "eval-ray",
        "algo": "SAC",
        "variant": "gen. distribution",
    },
    # --- DreamerV3 (only capacity_sampled was trained; eval run still pending) --
    {
		"path": "snapshots/20260718_141113_gen_battery_lin_so_only_17kW_dreamerv3",
		"eval-kind": "eval-ray",
		"algo": "DreamerV3",
		"variant": "trained on 17 kWh",
	},
    {
		"path": "snapshots/20260718_141218_gen_battery_lin_so_only_15kW_dreamerv3",
		"eval-kind": "eval-ray",
		"algo": "DreamerV3",
		"variant": "trained on 15 kWh",
	},
    {
		"path": "snapshots/20260718_140503_gen_battery_lin_so_all_dreamerv3",
		"eval-kind": "eval-ray",
		"algo": "DreamerV3",
		"variant": "5-way gen.",
	},
    {
        "path": "snapshots/20260716_175201_gen_battery_lin_capacity_sampled_dreamerv3",
        "eval-kind": "eval-ray",
        "algo": "DreamerV3",
        "variant": "gen. distribution",
    },
]

# Rule-based baselines: every strategy of this snapshot's eval_rbc run becomes one
# strip in the "RBC" x-axis group (drop entries here to trim the group). The
# rule-based eval is policy-independent — any snapshot with an eval_rbc run works.
RBC_SNAPSHOT = "snapshots/20260715_225505_gen_battery_lin_capacity_sampled_ppo_ctxt_div_fix"
RBC_STRATEGIES = (
    "do_nothing",
    "deficit_discharge",
    # "pv_surplus_charge",
    "self_coverage",
    "price_median",
    "price_median_scaled_0.25",
    "price_median_scaled_0.5",
    "price_median_scaled_0.75",
    "price_median_autarky",
)
SOURCES += [
    {"path": RBC_SNAPSHOT, "eval-kind": "eval-rbc", "algo": "RBC",
     "variant": strategy, "rbc-strategy": strategy}
    for strategy in RBC_STRATEGIES
]

# Which eval settings become the table columns. None = every setting discovered:
# held-out `eval_<i>` configs, numerically sorted (battery_lin_cap: eval_1 = BES
# 12 kWh, eval_2 = 16 kWh, eval_3 = 22 kWh); a flat STA layout appears as "default".
# Or an explicit tuple, e.g. ("eval_1", "eval_3").
EVAL_CFGS: tuple[str, ...] | None = None

# One exported table per metric: rows (algorithm, variant), columns the eval
# settings. Extend to also table e.g. "cum_E_kWh" and "cum_price_EUR".
METRICS = ("total_reward",)

# Optional display names for the LaTeX export's variant column, keyed by the raw
# variant label. Only affects the .tex body — the CSV and the tables displayed in
# this notebook keep the raw keys. Values are inserted as RAW LaTeX (no escaping —
# math like $\times$ is allowed, escape %, &, _ yourself); unlisted variants keep
# their raw label, LaTeX-escaped. Shorter names keep the table inside the A4 text
# block without shrinking the font further.
TEX_VARIANT_LABELS: dict[str, str] = {
    "do_nothing": "do nothing",
    "deficit_discharge": "deficit discharge",
    "self_coverage": "self coverage",
    "price_median": "price median",
    "price_median_scaled_0.25": "price median $\\times$0.25",
    "price_median_scaled_0.5": "price median $\\times$0.5",
    "price_median_scaled_0.75": "price median $\\times$0.75",
    "price_median_autarky": "price median autarky",
}


In [2]:
import json
import re
from pathlib import Path

import pandas as pd

EVAL_KINDS = ("eval-ray", "eval-sb", "eval-rbc")

# A held-out eval-config result directory ends in `_eval_<i>` (e.g. battery_lin_cap_eval_1);
# training-config directories (e.g. battery_lin_cap_3) do not match and are skipped.
_EVAL_CFG_DIR_PATTERN = re.compile(r"(?:^|_)eval_(\d+)$")


# Walk upwards from the cwd until the directory containing snapshots/ is found, so the
# notebook runs both from thesis_eval/ and from the repo root.
def _find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "snapshots").is_dir():
            return candidate
    raise FileNotFoundError("No 'snapshots/' directory found upwards of the cwd — run inside the repo.")


REPO_ROOT = _find_repo_root()


# Maps a runs/eval_* directory to its eval kind — mirrors the run-id scheme
# tools/snapshot/submit_snapshot.py builds from --kind.
def _run_dir_eval_kind(run_dir: Path) -> str:
    if run_dir.name.startswith("eval_rbc_"):
        return "eval-rbc"
    if run_dir.name.startswith("eval_sb_"):
        return "eval-sb"
    return "eval-ray"


# `eval_<i>` for a held-out eval-config result dir name, None for anything else.
def _cfg_dir_setting(dir_name: str) -> str | None:
    match = _EVAL_CFG_DIR_PATTERN.search(dir_name)
    return f"eval_{match.group(1)}" if match else None


# Ray/SB3 eval CSVs carry a sidecar JSON (same stem) with the run's trial seed + algorithm.
def _sidecar_seed_and_algorithm(csv_path: Path) -> tuple[int | None, str | None]:
    sidecar = csv_path.with_suffix(".json")
    if not sidecar.is_file():
        return None, None
    try:
        payload = json.loads(sidecar.read_text())
    except (json.JSONDecodeError, OSError):
        return None, None
    return payload.get("seed"), payload.get("algorithm")


# Rule-based runs record resolved_seed in an args.json above episodes.csv (directly
# above the per-strategy dir in the flat layout, one level further up with per-config
# dirs) — walk up towards the run dir and take the first hit.
def _rbc_resolved_seed(csv_path: Path, run_dir: Path) -> int | None:
    for candidate_dir in csv_path.parents:
        args_path = candidate_dir / "args.json"
        if args_path.is_file():
            try:
                return json.loads(args_path.read_text()).get("resolved_seed")
            except (json.JSONDecodeError, OSError):
                return None
        if candidate_dir == run_dir:
            return None
    return None


# Discovers one record per eval CSV of `source` across ALL eval settings kept by
# EVAL_CFGS (None = no filter). Both layouts are globbed; every record is tagged with
# its setting ("default" for the flat STA layout, `eval_<i>` for held-out GEN config
# dirs).
def _discover_source_records(source: dict) -> list[dict]:
    eval_kind = source["eval-kind"]
    if eval_kind not in EVAL_KINDS:
        raise ValueError(f"'eval-kind' must be one of {EVAL_KINDS}, got {eval_kind!r}")

    snapshot_dir = Path(source["path"])
    snapshot_dir = snapshot_dir if snapshot_dir.is_absolute() else REPO_ROOT / snapshot_dir
    if not snapshot_dir.is_dir():
        raise FileNotFoundError(f"Snapshot directory not found: {snapshot_dir}")

    run_dirs = [
        run_dir for run_dir in sorted((snapshot_dir / "runs").glob("eval_*"))
        if _run_dir_eval_kind(run_dir) == eval_kind
    ]
    if wanted_runs := source.get("runs"):
        run_dirs = [run_dir for run_dir in run_dirs if run_dir.name in set(wanted_runs)]
    if not run_dirs:
        raise FileNotFoundError(f"No '{eval_kind}' run directories found under {snapshot_dir / 'runs'}")

    records = []
    for run_dir in run_dirs:
        if eval_kind == "eval-rbc":
            # flat: .../<strategy>/episodes.csv — nested: .../<strategy>/<cfg>_eval_<i>/episodes.csv
            candidates = [
                (csv_path, "default", csv_path.parent.name)
                for csv_path in sorted(run_dir.glob("eval_results/*_rule_based/*/episodes.csv"))
            ] + [
                (csv_path, _cfg_dir_setting(csv_path.parent.name), csv_path.parent.parent.name)
                for csv_path in sorted(run_dir.glob("eval_results/*_rule_based/*/*/episodes.csv"))
            ]
            strategies = sorted({strategy for _, _, strategy in candidates})
            strategy_wanted = source.get("rbc-strategy")
            if strategy_wanted is None and len(strategies) > 1:
                raise ValueError(
                    f"Source '{source['algo']} — {source['variant']}': multiple rule-based strategies "
                    f"{strategies} under {run_dir.name} — set 'rbc-strategy' to pick one."
                )
            for csv_path, setting, strategy in candidates:
                if setting is None or (strategy_wanted is not None and strategy != strategy_wanted):
                    continue
                records.append({
                    "algo": source["algo"], "variant": source["variant"], "snapshot_dir": snapshot_dir,
                    "run": run_dir.name, "setting": setting, "seed": _rbc_resolved_seed(csv_path, run_dir),
                    "eval_algorithm": strategy, "csv": csv_path,
                })
        else:
            candidates = [
                (csv_path, "default")
                for csv_path in sorted(run_dir.glob("eval_results/*_eval/eval_*.csv"))
            ] + [
                (csv_path, _cfg_dir_setting(csv_path.parent.name))
                for csv_path in sorted(run_dir.glob("eval_results/*_eval/*/eval_*.csv"))
            ]
            for csv_path, setting in candidates:
                if setting is None:
                    continue  # training-config result dir — not a held-out eval config
                seed, algorithm = _sidecar_seed_and_algorithm(csv_path)
                records.append({
                    "algo": source["algo"], "variant": source["variant"], "snapshot_dir": snapshot_dir,
                    "run": run_dir.name, "setting": setting, "seed": seed,
                    "eval_algorithm": algorithm, "csv": csv_path,
                })

    available = sorted({record["setting"] for record in records})
    if EVAL_CFGS is not None:
        records = [record for record in records if record["setting"] in set(EVAL_CFGS)]
    if not records:
        raise FileNotFoundError(
            f"Source '{source['algo']} — {source['variant']}': no eval CSVs on setting(s) "
            f"{EVAL_CFGS} under {snapshot_dir} (available: {available or 'none'})."
        )
    return records


_source_keys = [(source["algo"], source["variant"]) for source in SOURCES]
if len(_source_keys) != len(set(_source_keys)):
    raise ValueError(f"Every source needs a unique (algo, variant) pair; got {_source_keys}")

# Pool a source's eval runs, but keep only the first CSV per (setting, seed): the
# same checkpoint evaluated with the same seed on the same setting replays the
# identical eval episodes, so a duplicate would double-count them. Seed-less CSVs are
# kept and flagged. A source without eval results is skipped with a warning.
eval_records: list[dict] = []
print(f"Repo root    : {REPO_ROOT}")
print(f"Eval settings: {'all discovered' if EVAL_CFGS is None else EVAL_CFGS}")
for source in SOURCES:
    try:
        discovered = sorted(_discover_source_records(source),
                            key=lambda record: (record["setting"], record["run"], record["csv"].name))
    except FileNotFoundError as error:
        print(f"\nWARNING: source '{source['algo']} — {source['variant']}' skipped — {error}")
        continue
    kept, seen = [], set()
    for record in discovered:
        if record["seed"] is not None and (record["setting"], record["seed"]) in seen:
            print(f"note: source '{source['algo']} — {source['variant']}': dropped {record['run']} / "
                  f"{record['csv'].name} on {record['setting']} (duplicate seed {record['seed']})")
            continue
        if record["seed"] is None:
            print(f"WARNING: source '{source['algo']} — {source['variant']}': no seed resolvable for "
                  f"{record['csv'].name} — kept, duplicate eval runs undetectable")
        else:
            seen.add((record["setting"], record["seed"]))
        kept.append(record)
    eval_records.extend(kept)

    strategy = f", strategy={source.get('rbc-strategy')}" if source["eval-kind"] == "eval-rbc" else ""
    print(f"\nSource '{source['algo']} — {source['variant']}' ({source['eval-kind']}{strategy}): {source['path']}")
    for record in kept:
        print(f"  {record['setting']:>8}  seed {str(record['seed']):>5}  <-  {record['run']} / {record['csv'].name}")

if not eval_records:
    raise FileNotFoundError("No eval CSVs discovered for any source — nothing to table.")

# Row order = first-appearance order in SOURCES, restricted to sources with results.
_algos_with_records = {record["algo"] for record in eval_records}
_variants_with_records = {record["variant"] for record in eval_records}
algo_labels = list(dict.fromkeys(source["algo"] for source in SOURCES if source["algo"] in _algos_with_records))
variant_labels = list(dict.fromkeys(source["variant"] for source in SOURCES if source["variant"] in _variants_with_records))


# Column order: the flat "default" setting first (if present), then eval_<i> numerically.
def _setting_sort_key(setting: str) -> tuple[int, int]:
    match = re.fullmatch(r"eval_(\d+)", setting)
    return (0, 0) if match is None else (1, int(match.group(1)))


setting_labels = sorted({record["setting"] for record in eval_records}, key=_setting_sort_key)
print(f"\nTable columns (eval settings): {setting_labels}")


Repo root    : /hkfs/home/haicore/iai/dj0397/AdvBuildingGym
Eval settings: all discovered

Source 'PPO — trained on 17 kWh' (eval-ray): snapshots/20260714_235444_gen_battery_lin_so_only_17kW_ppo
    eval_1  seed    42  <-  eval_20260715_101619 / eval_checkpoint_000058_20260715_103553.csv
    eval_2  seed    42  <-  eval_20260715_101619 / eval_checkpoint_000058_20260715_103618.csv
    eval_3  seed    42  <-  eval_20260715_101619 / eval_checkpoint_000058_20260715_103642.csv

Source 'PPO — trained on 15 kWh' (eval-ray): snapshots/20260714_235342_gen_battery_lin_so_only_15kW_ppo
    eval_1  seed    42  <-  eval_20260715_101616 / eval_checkpoint_000058_20260715_103410.csv
    eval_2  seed    42  <-  eval_20260715_101616 / eval_checkpoint_000058_20260715_103436.csv
    eval_3  seed    42  <-  eval_20260715_101616 / eval_checkpoint_000058_20260715_103500.csv

Source 'PPO — 5-way gen.' (eval-ray): snapshots/20260714_235542_gen_battery_lin_so_all_ppo
    eval_1  seed    42  <-  eval_20260715_10

## Load per-episode metrics

Every kept CSV is concatenated into one long-format frame, one row per eval episode,
tagged with its `algo`, training `variant`, `eval_cfg` (the setting) and the eval
run's `seed`. The preview below counts the episodes behind each table cell.


In [3]:
frames = []
for record in eval_records:
    csv_frame = pd.read_csv(record["csv"])
    present = [metric for metric in METRICS if metric in csv_frame.columns]
    missing = [metric for metric in METRICS if metric not in csv_frame.columns]
    if missing:
        print(f"note: source '{record['algo']} — {record['variant']}' {record['run']} lacks {missing} — "
              "skipped in those tables")
    frame = csv_frame[present].copy()
    frame["algo"] = record["algo"]
    frame["variant"] = record["variant"]
    frame["eval_cfg"] = record["setting"]
    frame["seed"] = record["seed"]
    frames.append(frame)

episodes = pd.concat(frames, ignore_index=True)
present_metrics = [metric for metric in METRICS if metric in episodes.columns]

# Episodes behind each table cell: rows (algo, variant), columns the eval settings.
episodes.groupby(["algo", "variant", "eval_cfg"]).size().unstack("eval_cfg").reindex(
    columns=setting_labels
).reindex(algo_labels, level="algo").reindex(variant_labels, level="variant")


eval_cfg                            eval_1  eval_2  eval_3
algo      variant                                         
PPO       trained on 17 kWh             10      10      10
          trained on 15 kWh             10      10      10
          5-way gen.                    10      10      10
          gen. distribution             10      10      10
SAC       trained on 17 kWh             10      10      10
          trained on 15 kWh             10      10      10
          5-way gen.                    10      10      10
          gen. distribution             10      10      10
DreamerV3 trained on 17 kWh             10      10      10
          trained on 15 kWh             10      10      10
          5-way gen.                    10      10      10
          gen. distribution             10      10      10
RBC       do_nothing                    10      10      10
          deficit_discharge             10      10      10
          self_coverage                 10      10      10
          price_median                  10      10      10
          price_median_scaled_0.25      10      10      10
          price_median_scaled_0.5       10      10      10
          price_median_scaled_0.75      10      10      10
          price_median_autarky          10      10      10

## Per-strip statistics across the eval settings — display and export

One table per metric: rows = (algorithm, variant), columns = the eval settings. The
cell below displays each table (2-decimal preview) and writes the CSV (full
precision) + LaTeX body into every source snapshot's `stat/` directory. The variant
column of the `.tex` uses the display names from `TEX_VARIANT_LABELS` (main-input
cell); the CSV and the displayed tables keep the raw variant keys.


In [4]:
from IPython.display import display

# Same algo tag as the catplot figure PDFs, so tables and figures pair up in stat/.
_compare_tag = re.sub(r"\W+", "_", "_vs_".join(algo_labels)).strip("_").lower()
_snapshot_dirs = sorted({record["snapshot_dir"] for record in eval_records}, key=str)

_STATS = ("count", "mean", "std", "median", "min", "max")


# Wide per-metric table: rows (algo, variant), columns (eval setting, stat).
def _metric_stats_wide(metric: str) -> pd.DataFrame:
    stats = episodes.groupby(["algo", "variant", "eval_cfg"])[metric].agg(list(_STATS))
    wide = stats.unstack("eval_cfg").swaplevel(axis=1)
    wide = wide.reindex(columns=pd.MultiIndex.from_product([setting_labels, _STATS]))
    return wide.reindex(algo_labels, level="algo").reindex(variant_labels, level="variant")


# LaTeX-escape the label columns (the RBC strategy names carry underscores).
def _latex_escape(text: str) -> str:
    return text.replace("&", r"\&").replace("%", r"\%").replace("_", r"\_")


# One table cell: "$mean \pm std$" at 2 decimals; bare mean when std is undefined
# (single-episode cell), "--" when a (row, setting) combination has no episodes.
def _latex_cell(mean: float, std: float | None = None) -> str:
    if pd.isna(mean):
        return "--"
    if std is not None and pd.notna(std):
        return f"${mean:.2f} \\pm {std:.2f}$"
    return f"${mean:.2f}$"


# Table body — rounding and ± assembly happen here, once, so the hand-written
# tabular skeleton that \input{}s this stays thin. \midrule separates the algorithm
# groups; the algorithm label is printed on its group's first row only. The file ends
# with \bottomrule ON PURPOSE: LaTeX inserts file-hook tokens at the end of an
# \input{}ed file, and after the last row's \\ those tokens start a new cell — a
# \bottomrule placed after \input{} in the skeleton therefore dies with
# "Misplaced \noalign". Inside the file it is safe; \end{tabular} may follow
# \input{} directly.
def _stats_latex_body(metric: str, wide: pd.DataFrame) -> str:
    lines = [
        f"% Auto-generated by snapshot_eval_stats_table_algo_compare.ipynb — booktabs body for '{metric}'",
        "% incl. trailing \\bottomrule (a \\bottomrule AFTER \\input{} would be 'Misplaced \\noalign').",
        "% Columns: algorithm & variant"
        + "".join(f" & {setting} mean+-std & {setting} median" for setting in setting_labels) + ".",
        "% Full precision, count, min and max: see the CSV with the same stem.",
        "% Verified skeleton:",
        f"%   \\begin{{tabular}}{{ll*{{{2 * len(setting_labels)}}}{{c}}}}",
        "%     \\toprule",
        "%     Algorithm & Variant & ... column heads ... \\\\",
        "%     \\midrule",
        "%     \\input{<this file>}",
        "%   \\end{tabular}",
    ]
    previous_algo = None
    for (algo, variant), row in wide.iterrows():
        if previous_algo is not None and algo != previous_algo:
            lines.append(r"\midrule")
        # TEX_VARIANT_LABELS values are raw LaTeX (no escaping); unmapped variants
        # fall back to their raw label, LaTeX-escaped.
        variant_cell = TEX_VARIANT_LABELS.get(variant, _latex_escape(str(variant)))
        cells = [_latex_escape(str(algo)) if algo != previous_algo else "", variant_cell]
        for setting in setting_labels:
            cells.append(_latex_cell(row[(setting, "mean")], row[(setting, "std")]))
            cells.append(_latex_cell(row[(setting, "median")]))
        lines.append(" & ".join(cells) + r" \\")
        previous_algo = algo
    lines.append(r"\bottomrule")
    return "\n".join(lines) + "\n"


for metric in present_metrics:
    wide = _metric_stats_wide(metric)
    export_frame = wide.copy()
    export_frame.columns = [f"{setting}_{stat}" for setting, stat in wide.columns]
    latex_body = _stats_latex_body(metric, wide)
    base_name = f"strip_stats_{metric}_evalcfg_compare_{_compare_tag}"
    for snapshot_dir in _snapshot_dirs:
        stat_dir = snapshot_dir / "stat"
        stat_dir.mkdir(parents=True, exist_ok=True)
        export_frame.to_csv(stat_dir / f"{base_name}.csv")
        (stat_dir / f"{base_name}.tex").write_text(latex_body)
        print(f"saved {stat_dir / base_name}.{{csv,tex}}")
    print(f"\n=== {metric} ===")
    display(wide.round(3))


saved /hkfs/home/haicore/iai/dj0397/AdvBuildingGym/snapshots/20260714_161830_gen_battery_lin_so_only_17kW/stat/strip_stats_total_reward_evalcfg_compare_ppo_vs_sac_vs_dreamerv3_vs_rbc.{csv,tex}
saved /hkfs/home/haicore/iai/dj0397/AdvBuildingGym/snapshots/20260714_162614_gen_battery_lin_so_all/stat/strip_stats_total_reward_evalcfg_compare_ppo_vs_sac_vs_dreamerv3_vs_rbc.{csv,tex}
saved /hkfs/home/haicore/iai/dj0397/AdvBuildingGym/snapshots/20260714_165943_gen_battery_lin_capacity_sampled/stat/strip_stats_total_reward_evalcfg_compare_ppo_vs_sac_vs_dreamerv3_vs_rbc.{csv,tex}
saved /hkfs/home/haicore/iai/dj0397/AdvBuildingGym/snapshots/20260714_235241_gen_battery_lin_so_only_15kW/stat/strip_stats_total_reward_evalcfg_compare_ppo_vs_sac_vs_dreamerv3_vs_rbc.{csv,tex}
saved /hkfs/home/haicore/iai/dj0397/AdvBuildingGym/snapshots/20260714_235342_gen_battery_lin_so_only_15kW_ppo/stat/strip_stats_total_reward_evalcfg_compare_ppo_vs_sac_vs_dreamerv3_vs_rbc.{csv,tex}
saved /hkfs/home/haicore/iai/dj03

eval_1                                \
                                    count    mean    std  median    min   
algo      variant                                                         
PPO       trained on 17 kWh            10  13.283  3.484  13.880  5.906   
          trained on 15 kWh            10  13.083  3.532  13.488  5.501   
          5-way gen.                   10  13.025  3.467  12.681  6.310   
          gen. distribution            10  13.082  3.445  12.936  6.017   
SAC       trained on 17 kWh            10   9.401  3.275   9.576  3.306   
          trained on 15 kWh            10   9.305  3.219   9.826  3.244   
          5-way gen.                   10  13.829  3.555  13.946  5.974   
          gen. distribution            10  13.412  3.422  13.815  6.042   
DreamerV3 trained on 17 kWh            10  15.294  3.980  15.453  6.529   
          trained on 15 kWh            10  13.546  4.041  13.764  4.550   
          5-way gen.                   10  14.596  4.088  15.322  5.515   
          gen. distribution            10  14.017  3.988  14.759  5.546   
RBC       do_nothing                   10   9.303  5.210  10.359 -0.375   
          deficit_discharge            10   6.813  4.379   8.055  0.660   
          self_coverage                10   6.750  4.453   7.974  0.398   
          price_median                 10   8.998  6.092   8.802  0.602   
          price_median_scaled_0.25     10  12.038  5.174  12.142  3.476   
          price_median_scaled_0.5      10  10.169  5.732  10.813  1.423   
          price_median_scaled_0.75     10   9.214  5.990   9.005  0.885   
          price_median_autarky         10  10.240  5.356  10.995  0.792   

                                           eval_2                         \
                                       max  count    mean    std  median   
algo      variant                                                          
PPO       trained on 17 kWh         18.142     10  15.453  3.351  16.330   
          trained on 15 kWh         17.980     10  15.223  3.271  15.478   
          5-way gen.                18.473     10  15.246  3.321  14.812   
          gen. distribution         18.231     10  15.192  3.320  14.908   
SAC       trained on 17 kWh         14.408     10  11.617  2.920  11.801   
          trained on 15 kWh         14.256     10  12.879  2.981  13.312   
          5-way gen.                19.051     10  15.780  3.449  16.133   
          gen. distribution         18.071     10  15.510  3.358  16.336   
DreamerV3 trained on 17 kWh         19.716     10  17.718  3.897  17.981   
          trained on 15 kWh         19.007     10  15.793  3.625  15.958   
          5-way gen.                19.455     10  16.267  3.647  16.505   
          gen. distribution         19.007     10  15.126  3.523  15.597   
RBC       do_nothing                18.364     10   9.303  5.210  10.359   
          deficit_discharge         14.793     10   6.122  4.147   7.104   
          self_coverage             14.753     10   6.051  4.220   6.993   
          price_median              20.687     10   9.185  6.356   8.223   
          price_median_scaled_0.25  22.022     10  13.632  4.975  13.087   
          price_median_scaled_0.5   21.251     10  11.376  5.695  11.670   
          price_median_scaled_0.75  20.889     10  10.116  6.038  10.088   
          price_median_autarky      20.266     10  10.533  5.382  11.036   

                                                  eval_3                 \
                                      min     max  count    mean    std   
algo      variant                                                         
PPO       trained on 17 kWh         8.020  19.392     10  17.543  3.224   
          trained on 15 kWh         7.712  18.954     10  17.548  3.069   
          5-way gen.                8.543  19.804     10  18.374  3.172   
          gen. distribution         8.043  19.416     10  18.205  3.215   
SAC       trained on 17 kWh        